# Psychische Gesundheit in Tech-Berufen: kurzer Durchlauf
Die wichtigsten Schritte der Analyse im Überblick. Alle Abbildungen und Kennzahlen erzeugt `python run_all.py`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

# projektordner in den suchpfad, damit src importiert werden kann
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import clean, cluster, features, load, profile, reduce
from src.utils import load_config

cfg = load_config()

## 1 Daten laden und filtern

In [2]:
raw, questions = load.load_raw(cfg)
df, flog = clean.filter_sample(raw)
df, clog, tables = clean.clean_all(df, cfg)
print(raw.shape, flog)
tables["gender_mapping"].groupby("kategorie").n.sum()

(1433, 63) {'n_roh': 1433, 'n_nach_selbststaendige': 1146, 'n_selbststaendige_ausgeschlossen': 287, 'n_tech_unternehmen': 883, 'n_tech_rolle_in_nicht_tech': 248, 'n_ohne_tech_bezug_ausgeschlossen': 15, 'n_analyse': 1131}


kategorie
divers/andere     23
keine Angabe       4
maennlich        841
weiblich         263
Name: n, dtype: int64

## 2 Feature Engineering: ordinale Kodierung und Indizes

In [3]:
X, idx, flog2 = features.build_workplace_matrix(df, cfg)
print(X.shape)
idx.describe().round(2)

(1131, 16)


,support_index,openness_index,stigma_index,uncertainty_index
count,1131.00,1131.00,1131.00,1131.00
mean,0.46,0.50,0.39,0.31
std,0.25,0.35,0.19,0.20
min,0.00,0.00,0.00,0.00
25%,0.30,0.25,0.25,0.14
50%,0.40,0.50,0.40,0.29
75%,0.60,0.75,0.50,0.43
max,1.00,1.00,1.00,1.00


## 3 Dimensionsreduktion (PCA) und Clustering (k-Means)

In [4]:
Z = reduce.scale(X, cfg["scaling"]["method"])
p = reduce.fit_pca(Z, list(X.columns), cfg["reduction"]["pca_variance_threshold"], cfg["random_state"])
S = p["scores"][:, :p["n_components"]]
labels, _ = cluster.fit_model("kmeans", S, cfg["clustering"]["final_k"], cfg)
labels = cluster.relabel_by_size(labels)
print("Komponenten:", p["n_components"], "| kum. Varianz:", round(p["cum"][p["n_components"]-1], 3))
print(cluster.internal_metrics(S, labels))
profile.cluster_sizes(labels)

Komponenten: 9 | kum. Varianz: 0.835
{'silhouette': 0.15839841279384737, 'davies_bouldin': 1.926518462676301, 'calinski_harabasz': 238.0146546137012, 'min_share': 0.20601237842617154}


,cluster,n,anteil
0,1,350,0.309461
1,2,277,0.244916
2,3,271,0.239611
3,4,233,0.206012


## 4 Clusterprofile

In [5]:
profile.raw_means(pd.concat([X, idx[["support_index","openness_index","stigma_index"]]], axis=1), labels).round(2).T

,1,2,3,4,gesamt
mh_benefits,0.51,0.19,0.93,0.85,0.60
care_options_known,0.40,0.30,0.58,0.68,0.48
wellness_discussed,0.09,0.08,0.12,0.80,0.24
mh_resources,0.22,0.14,0.49,0.86,0.40
anonymity_protected,0.59,0.44,0.62,0.78,0.60
leave_ease,0.72,0.37,0.40,0.74,0.56
mh_neg_consequence_employer,0.86,0.31,0.35,0.83,0.60
ph_neg_consequence_employer,0.96,0.71,0.77,0.94,0.85
comfort_coworkers,0.65,0.19,0.26,0.68,0.45
comfort_supervisor,0.81,0.22,0.28,0.83,0.54


Ausgewählte Abbildungen (erzeugt von `run_all.py`):

![Clustermittelwerte je Merkmal](../figures/abb_16_cluster_heatmap.png)

![Zweidimensionale Einbettungen nach Cluster](../figures/abb_10_einbettungen_cluster.png)

![Cluster im Indexraum](../figures/abb_20_indexraum.png)

## 5 Zentrale Kennzahlen aus `results/metrics.json`

In [6]:
m = json.loads((ROOT / "results" / "metrics.json").read_text(encoding="utf-8"))
{k: m["iteration3"]["final"][k] for k in ["silhouette", "davies_bouldin", "calinski_harabasz", "min_share"]} | {"bootstrap_ari": m["stabilitaet"]["final_ari_mittel"]}

{'silhouette': 0.158398,
 'davies_bouldin': 1.92652,
 'calinski_harabasz': 238.015,
 'min_share': 0.206012,
 'bootstrap_ari': 0.846285}